In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    BEHAVIOR_FEATURES_FILE,
    GROUND_TRUTH_PARQUET,
    DATA_DICTIONARY_FILE,
)


# ============================================================
# HELPERS
# ============================================================

def db_path(path):
    return Path(path).resolve().as_posix()


# ============================================================
# INPUT CHECK
# ============================================================

required_files = {
    "Behavior features": BEHAVIOR_FEATURES_FILE,
    "Ground truth": GROUND_TRUTH_PARQUET,
}

missing = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not Path(path).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing)
    )

print("✓ Required input files found.")


# ============================================================
# PATHS
# ============================================================

features_path = db_path(
    BEHAVIOR_FEATURES_FILE
)

ground_truth_path = db_path(
    GROUND_TRUTH_PARQUET
)


# ============================================================
# READ ACTUAL SCHEMAS
# ============================================================

con = duckdb.connect()

feature_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{features_path}'
    )
""").df()

gt_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{ground_truth_path}'
    )
""").df()

con.close()


# ============================================================
# BEHAVIOR FEATURE DESCRIPTIONS
# ============================================================

descriptions = {

    # IDENTIFIERS / TIME
    "user":
        "Unique user identifier.",

    "day":
        "Date of the aggregated user activity.",

    "day_of_week":
        "Numeric representation of the day of the week.",

    "is_weekend":
        "Indicates whether the day is a weekend.",


    # EMAIL
    "email_count":
        "Total number of email events generated by the user during the day.",

    "external_email_events":
        "Number of email events containing at least one recipient outside the organization domain.",

    "total_email_bytes":
        "Total size of email messages generated during the day, in bytes.",

    "attachment_count":
        "Total number of email attachments during the day.",


    # USB / FILE
    "usb_connect_count":
        "Number of removable USB device connection events during the day.",

    "usb_disconnect_count":
        "Number of removable USB device disconnection events during the day.",

    "files_copied_to_usb":
        "Number of files copied to removable media during the day.",

    "unique_file_extensions":
        "Number of unique file extensions copied to removable media during the day.",


    # HTTP
    "http_request_count":
        "Total number of HTTP requests generated by the user during the day.",

    "unique_domains":
        "Number of unique web domains accessed during the day.",


    # LOGON
    "logon_count":
        "Number of user logon events during the day.",

    "first_logon":
        "Timestamp of the user's first logon event during the day.",

    "last_logoff":
        "Timestamp of the user's last logoff event during the day.",

    "first_logon_minute":
        "First logon time represented as minutes since midnight.",

    "last_logoff_minute":
        "Last logoff time represented as minutes since midnight.",


    # BASELINE
    "baseline_workdays":
        "Number of valid workdays used to construct the user's historical baseline.",

    "email_count_median":
        "Median daily email count during the baseline period.",

    "email_count_p95":
        "95th percentile of daily email count during the baseline period.",

    "external_email_median":
        "Median daily count of external email events during the baseline period.",

    "external_email_p95":
        "95th percentile of daily external email events during the baseline period.",

    "email_bytes_median":
        "Median daily total email size in bytes during the baseline period.",

    "email_bytes_p95":
        "95th percentile of daily total email size during the baseline period.",

    "attachment_median":
        "Median daily email attachment count during the baseline period.",

    "attachment_p95":
        "95th percentile of daily email attachment count during the baseline period.",

    "usb_connect_median":
        "Median daily USB connection count during the baseline period.",

    "usb_connect_p95":
        "95th percentile of daily USB connection count during the baseline period.",

    "usb_files_median":
        "Median daily number of files copied to removable media during the baseline period.",

    "usb_files_p95":
        "95th percentile of daily number of files copied to removable media during the baseline period.",

    "http_requests_median":
        "Median daily HTTP request count during the baseline period.",

    "http_requests_p95":
        "95th percentile of daily HTTP request count during the baseline period.",

    "unique_domains_median":
        "Median daily number of unique web domains during the baseline period.",

    "unique_domains_p95":
        "95th percentile of daily unique web domains during the baseline period.",

    "first_logon_median":
        "Median first logon time in minutes since midnight during the baseline period.",

    "first_logon_p05":
        "5th percentile of first logon time during the baseline period.",

    "first_logon_p95":
        "95th percentile of first logon time during the baseline period.",

    "last_logoff_median":
        "Median last logoff time in minutes since midnight during the baseline period.",

    "last_logoff_p05":
        "5th percentile of last logoff time during the baseline period.",

    "last_logoff_p95":
        "95th percentile of last logoff time during the baseline period.",


    # BASELINE RATIOS
    "email_p95_ratio":
        "Ratio of current daily email activity to the user's baseline 95th percentile.",

    "external_email_p95_ratio":
        "Ratio of current external email activity to the user's baseline 95th percentile.",

    "usb_connect_p95_ratio":
        "Ratio of current USB connection activity to the user's baseline 95th percentile.",

    "usb_files_p95_ratio":
        "Ratio of files copied to USB to the user's baseline 95th percentile.",

    "http_p95_ratio":
        "Ratio of current HTTP activity to the user's baseline 95th percentile.",

    "domains_p95_ratio":
        "Ratio of current unique-domain activity to the user's baseline 95th percentile.",


    # ABOVE-BASELINE FLAGS
    "email_above_p95":
        "Indicates whether daily email activity exceeds the user's baseline 95th percentile.",

    "external_email_above_p95":
        "Indicates whether external email activity exceeds the user's baseline 95th percentile.",

    "usb_connect_above_p95":
        "Indicates whether USB connection activity exceeds the user's baseline 95th percentile.",

    "usb_files_above_p95":
        "Indicates whether files copied to USB exceed the user's baseline 95th percentile.",

    "http_above_p95":
        "Indicates whether HTTP activity exceeds the user's baseline 95th percentile.",

    "domains_above_p95":
        "Indicates whether unique-domain activity exceeds the user's baseline 95th percentile.",


    # WORKING-TIME DEVIATION
    "early_logon_minutes":
        "Number of minutes the user's first logon occurred earlier than the baseline lower range.",

    "late_logoff_minutes":
        "Number of minutes the user's last logoff occurred later than the baseline upper range.",


    # CHANNEL ACTIVITY
    "email_active":
        "Indicates whether the email channel was active during the day.",

    "usb_active":
        "Indicates whether USB or removable-media activity was observed during the day.",

    "http_active":
        "Indicates whether HTTP activity was observed during the day.",


    # DAILY ABNORMALITY
    "daily_abnormal_feature_count":
        "Number of behavioral features exceeding their baseline thresholds during the day.",

    "abnormal_day":
        "Indicates whether at least one behavioral feature exceeded its baseline threshold during the day.",


    # 7-DAY FEATURES
    "external_emails_7d":
        "Total external email events observed over the current and previous six User-Day records.",

    "usb_files_7d":
        "Total files copied to removable media over the current and previous six User-Day records.",

    "http_requests_7d":
        "Total HTTP requests observed over the current and previous six User-Day records.",

    "abnormal_days_7d":
        "Number of abnormal User-Day records within the current and previous six records.",


    # 14-DAY FEATURES
    "external_emails_14d":
        "Total external email events observed over the current and previous thirteen User-Day records.",

    "usb_files_14d":
        "Total files copied to removable media over the current and previous thirteen User-Day records.",

    "http_requests_14d":
        "Total HTTP requests observed over the current and previous thirteen User-Day records.",

    "abnormal_days_14d":
        "Number of abnormal User-Day records within the current and previous thirteen records.",


    # 30-DAY FEATURES
    "usb_files_30d":
        "Total files copied to removable media over the current and previous twenty-nine User-Day records.",

    "external_emails_30d":
        "Total external email events observed over the current and previous twenty-nine User-Day records.",

    "abnormal_days_30d":
        "Number of abnormal User-Day records within the current and previous twenty-nine records.",


    # CHANNEL DIVERSITY
    "activity_channel_diversity_7d":
        "Number of Email, USB, and HTTP activity channels active at least once within the current and previous six User-Day records. This feature measures channel diversity, not channel rotation.",
}


# ============================================================
# FEATURE CATEGORY
# ============================================================

def category(column):

    if column in {
        "user",
        "day",
        "day_of_week",
        "is_weekend",
    }:
        return "Identifier / Time"

    if column in {
        "baseline_workdays",
        "email_count_median",
        "email_count_p95",
        "external_email_median",
        "external_email_p95",
        "email_bytes_median",
        "email_bytes_p95",
        "attachment_median",
        "attachment_p95",
        "usb_connect_median",
        "usb_connect_p95",
        "usb_files_median",
        "usb_files_p95",
        "http_requests_median",
        "http_requests_p95",
        "unique_domains_median",
        "unique_domains_p95",
        "first_logon_median",
        "first_logon_p05",
        "first_logon_p95",
        "last_logoff_median",
        "last_logoff_p05",
        "last_logoff_p95",
    }:
        return "User Baseline"

    if column.endswith("_ratio"):
        return "Baseline Ratio"

    if column.endswith("_above_p95"):
        return "Behavioral Flag"

    if column in {
        "daily_abnormal_feature_count",
        "abnormal_day",
        "early_logon_minutes",
        "late_logoff_minutes",
    }:
        return "Behavioral"

    if (
        column.endswith("_7d")
        or column.endswith("_14d")
        or column.endswith("_30d")
    ):
        return "Temporal / Rolling"

    if column in {
        "email_active",
        "usb_active",
        "http_active",
    }:
        return "Channel Activity"

    if "email" in column or "attachment" in column:
        return "Email"

    if "usb" in column or "file" in column:
        return "USB / File"

    if "http" in column or "domain" in column:
        return "HTTP"

    if "logon" in column or "logoff" in column:
        return "Logon"

    return "Derived"


# ============================================================
# BUILD BEHAVIOR FEATURE DICTIONARY
# ============================================================

dictionary = feature_schema[
    [
        "column_name",
        "column_type",
    ]
].copy()

dictionary.columns = [
    "Feature",
    "Data Type",
]

dictionary["Description"] = (
    dictionary["Feature"]
    .map(descriptions)
)

dictionary["Category"] = (
    dictionary["Feature"]
    .apply(category)
)

dictionary["Dataset"] = (
    BEHAVIOR_FEATURES_FILE.name
)


# ============================================================
# GROUND TRUTH DICTIONARY
# ============================================================

gt_descriptions = {

    "user":
        "Unique identifier of the malicious user.",

    "scenario":
        "CERT r4.2 malicious scenario number.",

    "scenario_name":
        "Descriptive name assigned to the CERT attack scenario.",

    "details":
        "Original CERT incident-observables filename.",

    "attack_start":
        "Start timestamp of the malicious incident according to CERT ground truth.",

    "attack_end":
        "End timestamp of the malicious incident according to CERT ground truth.",

    "duration_days":
        "Duration of the malicious incident in days.",

    "exfiltration_relevant":
        "Indicates whether the incident belongs to a scenario relevant to the project's data-exfiltration experiments.",

    "source":
        "Source of the ground-truth record.",

    "is_synthetic":
        "Indicates whether the incident is synthetic. Original CERT incidents use 0.",
}


gt_dictionary = gt_schema[
    [
        "column_name",
        "column_type",
    ]
].copy()

gt_dictionary.columns = [
    "Feature",
    "Data Type",
]

gt_dictionary["Description"] = (
    gt_dictionary["Feature"]
    .map(gt_descriptions)
)

gt_dictionary["Category"] = "Ground Truth"

gt_dictionary["Dataset"] = (
    GROUND_TRUTH_PARQUET.name
)


# ============================================================
# VALIDATE DICTIONARY COMPLETENESS
# ============================================================

missing_behavior_descriptions = (
    dictionary[
        dictionary["Description"].isna()
    ]
)

missing_gt_descriptions = (
    gt_dictionary[
        gt_dictionary["Description"].isna()
    ]
)

assert len(dictionary) == 73, (
    f"Expected 73 behavior features, "
    f"found {len(dictionary)}."
)

assert missing_behavior_descriptions.empty, (
    "Behavior features missing descriptions:\n"
    + "\n".join(
        missing_behavior_descriptions[
            "Feature"
        ].tolist()
    )
)

assert missing_gt_descriptions.empty, (
    "Ground-truth fields missing descriptions:\n"
    + "\n".join(
        missing_gt_descriptions[
            "Feature"
        ].tolist()
    )
)


# ============================================================
# SAVE EXCEL
# ============================================================

with pd.ExcelWriter(
    DATA_DICTIONARY_FILE,
    engine="openpyxl",
) as writer:

    dictionary.to_excel(
        writer,
        sheet_name="Behavior Features",
        index=False,
    )

    gt_dictionary.to_excel(
        writer,
        sheet_name="Ground Truth",
        index=False,
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("✅ DATA DICTIONARY: PASS")
print("=" * 70)

print(
    f"\nBehavior features documented : "
    f"{len(dictionary)}"
)

print(
    f"Ground-truth fields documented: "
    f"{len(gt_dictionary)}"
)

print(
    "\nMissing behavior descriptions : "
    f"{len(missing_behavior_descriptions)}"
)

print(
    "Missing ground-truth descriptions: "
    f"{len(missing_gt_descriptions)}"
)

print(
    f"\nSaved to:\n"
    f"{DATA_DICTIONARY_FILE}"
)

print("\nBehavior feature preview:")
display(dictionary.head(10))

print("\nGround-truth preview:")
display(gt_dictionary)

✓ Required input files found.

✅ DATA DICTIONARY: PASS

Behavior features documented : 73
Ground-truth fields documented: 10

Missing behavior descriptions : 0
Missing ground-truth descriptions: 0

Saved to:
C:\Users\HP\Documents\FYP\Data_engineering\docs\data_dictionary.xlsx

Behavior feature preview:


,Feature,Data Type,Description,Category,Dataset
0,user,VARCHAR,Unique user identifier.,Identifier / Time,user_day_behavior_features_v1.parquet
1,day,DATE,Date of the aggregated user activity.,Identifier / Time,user_day_behavior_features_v1.parquet
2,day_of_week,BIGINT,Numeric representation of the day of the week.,Identifier / Time,user_day_behavior_features_v1.parquet
3,is_weekend,INTEGER,Indicates whether the day is a weekend.,Identifier / Time,user_day_behavior_features_v1.parquet
4,email_count,BIGINT,Total number of email events generated by the ...,Email,user_day_behavior_features_v1.parquet
5,external_email_events,DOUBLE,Number of email events containing at least one...,Email,user_day_behavior_features_v1.parquet
6,total_email_bytes,DOUBLE,Total size of email messages generated during ...,Email,user_day_behavior_features_v1.parquet
7,attachment_count,DOUBLE,Total number of email attachments during the day.,Email,user_day_behavior_features_v1.parquet
8,usb_connect_count,DOUBLE,Number of removable USB device connection even...,USB / File,user_day_behavior_features_v1.parquet
9,usb_disconnect_count,DOUBLE,Number of removable USB device disconnection e...,USB / File,user_day_behavior_features_v1.parquet



Ground-truth preview:


,Feature,Data Type,Description,Category,Dataset
0,user,VARCHAR,Unique identifier of the malicious user.,Ground Truth,ground_truth_r42.parquet
1,scenario,BIGINT,CERT r4.2 malicious scenario number.,Ground Truth,ground_truth_r42.parquet
2,scenario_name,VARCHAR,Descriptive name assigned to the CERT attack s...,Ground Truth,ground_truth_r42.parquet
3,details,VARCHAR,Original CERT incident-observables filename.,Ground Truth,ground_truth_r42.parquet
4,attack_start,TIMESTAMP_NS,Start timestamp of the malicious incident acco...,Ground Truth,ground_truth_r42.parquet
5,attack_end,TIMESTAMP_NS,End timestamp of the malicious incident accord...,Ground Truth,ground_truth_r42.parquet
6,duration_days,DOUBLE,Duration of the malicious incident in days.,Ground Truth,ground_truth_r42.parquet
7,exfiltration_relevant,INTEGER,Indicates whether the incident belongs to a sc...,Ground Truth,ground_truth_r42.parquet
8,source,VARCHAR,Source of the ground-truth record.,Ground Truth,ground_truth_r42.parquet
9,is_synthetic,BIGINT,Indicates whether the incident is synthetic. O...,Ground Truth,ground_truth_r42.parquet
